In [ ]:
!pip -q install -U transformers accelerate sentencepiece huggingface_hub bitsandbytes pandas openpyxl tqdm


from huggingface_hub import login



import os, time, re
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


XLSX_PATH = "/content/Dataset_550LLM.xlsx"

OUT_CSV = "/content/llama2_7b_chat_zeroshoot_output.csv"
PROGRESS_CSV = "/content/llama2_7b_chat_zeroshoot_progress.csv"


MODEL_ID = "meta-llama/Llama-2-7b-chat-hf"


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(" Model loaded on:", model.device)


def make_zero_shot_en(user_text: str) -> str:
    return (
        "You are a supportive mental-health assistant. "
        "Respond safely, empathetically, and practically. Keep it concise.\n\n"
        f"User: {user_text}\nAssistant:"
    )

def make_zero_shot_hi(user_text: str) -> str:
    return (
        "आप एक सहायक मानसिक-स्वास्थ्य सहायक हैं। "
        "सुरक्षित, सहानुभूतिपूर्ण और व्यावहारिक जवाब दें। संक्षिप्त रखें।\n\n"
        f"यूज़र: {user_text}\nसहायक:"
    )


@torch.inference_mode()
def generate_response(prompt: str,
                      max_new_tokens: int = 220,
                      temperature: float = 0.7,
                      top_p: float = 0.9) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
    )

    text = tokenizer.decode(out[0], skip_special_tokens=True)


    if "Assistant:" in text:
        text = text.split("Assistant:", 1)[-1].strip()
    return text.strip()


df = pd.read_excel(XLSX_PATH)


required = {"PromptID", "English", "Hindi"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}. Found: {df.columns.tolist()}")

print(" Rows:", len(df))
print(" Columns:", df.columns.tolist())


done_ids = set()

if os.path.exists(PROGRESS_CSV):
    prog = pd.read_csv(PROGRESS_CSV, encoding="utf-8-sig")
    if "PromptID" in prog.columns:
        done_ids = set(prog["PromptID"].astype(str).tolist())
    print(f" Resume: checkpoint found. Done = {len(done_ids)}")
else:
    print(" Fresh run: no checkpoint found.")

results = []
if os.path.exists(OUT_CSV):
    old = pd.read_csv(OUT_CSV, encoding="utf-8-sig")
    results = old.to_dict("records")
    print(f" Existing output loaded: {len(results)} rows")


SAVE_EVERY = 1
SLEEP_BETWEEN = 0.1

for _, row in tqdm(df.iterrows(), total=len(df)):
    pid = str(row["PromptID"])
    if pid in done_ids:
        continue

    en_text = "" if pd.isna(row["English"]) else str(row["English"])
    hi_text = "" if pd.isna(row["Hindi"]) else str(row["Hindi"])

    en_prompt = make_zero_shot_en(en_text)
    hi_prompt = make_zero_shot_hi(hi_text)

    try:
        en_resp = generate_response(en_prompt)
    except Exception as e:
        en_resp = f"[ERROR] {type(e).__name__}: {e}"

    time.sleep(SLEEP_BETWEEN)

    try:
        hi_resp = generate_response(hi_prompt)
    except Exception as e:
        hi_resp = f"[ERROR] {type(e).__name__}: {e}"

    results.append({
        "PromptID": pid,
        "English": en_text,
        "Hindi": hi_text,
        "EN_ZeroShot_Prompt": en_prompt,
        "EN_ZeroShot_Response": en_resp,
        "HI_ZeroShot_Prompt": hi_prompt,
        "HI_ZeroShot_Response": hi_resp,
        "Model": MODEL_ID,
    })

    done_ids.add(pid)


    if len(results) % SAVE_EVERY == 0:
        out_df = pd.DataFrame(results)
        out_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

        pd.DataFrame({"PromptID": sorted(list(done_ids))}).to_csv(
            PROGRESS_CSV, index=False, encoding="utf-8-sig"
        )

print(" DONE")
print("Output:", OUT_CSV)
print("Progress:", PROGRESS_CSV)


from google.colab import files
files.download(OUT_CSV)
files.download(PROGRESS_CSV)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Model loaded on: cuda:0
✅ Rows: 554
✅ Columns: ['PromptID', 'English', 'Hindi']
🟦 Fresh run: no checkpoint found.


100%|██████████| 554/554 [3:50:00<00:00, 24.91s/it]

✅ DONE
Output: /content/llama2_7b_chat_zeroshoot_output.csv
Progress: /content/llama2_7b_chat_zeroshoot_progress.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>